# Classic Algorithms from Scratch

KNN, K-means, decision trees, and Naive Bayes are asked at virtually every FAANG-adjacent ML coding round. The question is never "do you know sklearn" — it's "can you implement it without sklearn?" This note covers all four with vectorized implementations and complexity analysis.

## What Interviewers Test
- Vectorized pairwise distance computation (the `a²-2ab+b²` trick)
- K-means++ initialization vs random init — and why it matters
- Gini impurity derivation and recursive tree splitting
- Log-space Naive Bayes to avoid underflow
- Time/space complexity of each algorithm
- Which algorithm each company tends to ask (see interview tips below)

## Complexity Reference Table

| Algorithm | Train | Predict | Space | Notes |
|---|---|---|---|---|
| KNN | O(1) | O(n·d) per query | O(n·d) | No training; brute-force predict |
| K-means | O(I·k·n·d) | O(k·d) | O((n+k)·d) | I = iterations |
| Decision Tree | O(n·d·log n) per node | O(depth) | O(nodes) | Depth ≤ log₂(n) balanced |
| Naive Bayes | O(n·d) | O(k·d) | O(k·d) | k = classes |

## 1. KNN — Vectorized Pairwise Distances

The key trick for vectorized Euclidean distance:

$$\|a - b\|^2 = a^T a - 2a^T b + b^T b$$

This avoids a double loop and computes all pairwise distances as one matrix expression.

> 💡 **Interview Tip:** Google and Meta frequently ask KNN. The vectorization trick is what separates candidates. After implementing, they'll ask: *"How would you scale this to millions of points?"* — answer: ANN (HNSW, FAISS), LSH, or KD-trees.


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Vectorized pairwise squared Euclidean distances ---
def pairwise_sq_distances(A, B):
    """
    Compute squared Euclidean distances between rows of A (m,d) and B (n,d).
    Uses: ||a-b||^2 = ||a||^2 - 2*a·b + ||b||^2
    Returns: (m, n) matrix
    """
    # (m, 1) - (m, n) + (1, n) → broadcasts to (m, n)
    sq_A = np.sum(A**2, axis=1, keepdims=True)   # (m, 1)
    sq_B = np.sum(B**2, axis=1, keepdims=True).T  # (1, n)
    dot   = A @ B.T                               # (m, n)
    return sq_A - 2 * dot + sq_B

class KNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        dists = pairwise_sq_distances(X, self.X_train)   # (m, n_train)
        top_k_idx = np.argpartition(dists, self.k, axis=1)[:, :self.k]  # (m, k)
        top_k_labels = self.y_train[top_k_idx]            # (m, k)
        # Majority vote per row
        from scipy import stats
        return stats.mode(top_k_labels, axis=1, keepdims=False)[0]

# --- Test ---
X, y = make_classification(n_samples=400, n_features=8, n_informative=5, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

knn_scratch = KNN(k=5).fit(X_tr, y_tr)
knn_sk      = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)

preds_scratch = knn_scratch.predict(X_te)
preds_sk      = knn_sk.predict(X_te)

acc_scratch = np.mean(preds_scratch == y_te)
acc_sk      = np.mean(preds_sk == y_te)
print(f"KNN from scratch accuracy: {acc_scratch:.3f}")
print(f"KNN sklearn accuracy:      {acc_sk:.3f}")
print(f"Predictions agree: {np.mean(preds_scratch == preds_sk):.3f}")


## 2. K-Means with K-Means++ Initialization

**Random init** can lead to poor local optima. **K-means++** spreads initial centroids:
1. Pick first centroid randomly
2. Pick each subsequent centroid with probability proportional to squared distance from nearest existing centroid

> 💡 **Interview Tip:** Amazon and Apple ask K-means. After implementation they ask: *"How do you choose k?"* — answer: elbow method on inertia, silhouette score, or domain knowledge. Follow-up: *"How does K-means fail?"* — answer: assumes spherical clusters, sensitive to scale, can converge to local optima.


In [ ]:
class KMeans:
    def __init__(self, k=3, n_iter=100, init='kmeans++'):
        self.k = k
        self.n_iter = n_iter
        self.init = init

    def _init_centroids(self, X):
        n = X.shape[0]
        if self.init == 'random':
            idx = np.random.choice(n, self.k, replace=False)
            return X[idx].copy()
        # K-means++ initialization
        centroids = [X[np.random.randint(n)]]
        for _ in range(self.k - 1):
            # Squared distance to nearest centroid for each point
            dists = pairwise_sq_distances(X, np.array(centroids))  # (n, len(centroids))
            min_dists = dists.min(axis=1)                            # (n,)
            probs = min_dists / min_dists.sum()
            next_idx = np.random.choice(n, p=probs)
            centroids.append(X[next_idx])
        return np.array(centroids)  # (k, d)

    def fit(self, X):
        self.centroids_ = self._init_centroids(X)
        for _ in range(self.n_iter):
            # E-step: assign clusters
            dists = pairwise_sq_distances(X, self.centroids_)  # (n, k)
            labels = dists.argmin(axis=1)                       # (n,)
            # M-step: update centroids
            new_centroids = np.array([
                X[labels == k].mean(axis=0) if (labels == k).any() else self.centroids_[k]
                for k in range(self.k)
            ])
            if np.allclose(new_centroids, self.centroids_):
                break
            self.centroids_ = new_centroids
        self.labels_ = labels
        self.inertia_ = np.sum(dists[np.arange(len(X)), labels])
        return self

    def predict(self, X):
        dists = pairwise_sq_distances(X, self.centroids_)
        return dists.argmin(axis=1)

# --- Test ---
from sklearn.cluster import KMeans as SkKMeans
X_cl, true_labels = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

km_scratch = KMeans(k=4, init='kmeans++').fit(X_cl)
km_sk      = SkKMeans(n_clusters=4, init='k-means++', n_init=1, random_state=42).fit(X_cl)

print(f"Scratch inertia: {km_scratch.inertia_:.2f}")
print(f"sklearn inertia: {km_sk.inertia_:.2f}")
print(f"Unique clusters found (scratch): {len(np.unique(km_scratch.labels_))}")


## 3. Decision Tree for Classification

**Gini impurity:** $G = 1 - \sum_k p_k^2$  
**Information gain:** $IG = G_{parent} - \frac{n_L}{n} G_L - \frac{n_R}{n} G_R$

> 💡 **Interview Tip:** The recursive structure is what interviewers want to see. Start by writing the recursive `_build` function signature before any code. Follow-up questions: *"Why Gini over entropy?"* (answer: computationally cheaper, similar performance), *"How do you prevent overfitting?"* (max_depth, min_samples_split, pruning).


In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.tree_ = self._build(X, y, depth=0)
        return self

    def _gini(self, y):
        if len(y) == 0:
            return 0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return 1 - np.sum(probs**2)

    def _best_split(self, X, y):
        n, d = X.shape
        best_gain, best_feat, best_thresh = -1, None, None
        parent_gini = self._gini(y)
        for j in range(d):
            thresholds = np.unique(X[:, j])
            for thresh in thresholds:
                left  = y[X[:, j] <= thresh]
                right = y[X[:, j] >  thresh]
                if len(left) == 0 or len(right) == 0:
                    continue
                gain = parent_gini - (len(left)/n)*self._gini(left) - (len(right)/n)*self._gini(right)
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, j, thresh
        return best_feat, best_thresh

    def _build(self, X, y, depth):
        # Stopping criteria
        if depth >= self.max_depth or len(y) < self.min_samples_split or len(np.unique(y)) == 1:
            counts = np.bincount(y.astype(int), minlength=len(self.classes_))
            return {'leaf': True, 'class': np.argmax(counts)}
        feat, thresh = self._best_split(X, y)
        if feat is None:
            counts = np.bincount(y.astype(int), minlength=len(self.classes_))
            return {'leaf': True, 'class': np.argmax(counts)}
        mask = X[:, feat] <= thresh
        return {
            'leaf': False, 'feat': feat, 'thresh': thresh,
            'left':  self._build(X[mask],  y[mask],  depth+1),
            'right': self._build(X[~mask], y[~mask], depth+1),
        }

    def _predict_one(self, x, node):
        if node['leaf']:
            return node['class']
        if x[node['feat']] <= node['thresh']:
            return self._predict_one(x, node['left'])
        return self._predict_one(x, node['right'])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree_) for x in X])

# --- Test ---
from sklearn.tree import DecisionTreeClassifier as SkTree
X_small, y_small = make_classification(n_samples=300, n_features=6, n_informative=4, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_small, y_small, test_size=0.2, random_state=42)

dt_scratch = DecisionTreeClassifier(max_depth=4).fit(X_tr, y_tr)
dt_sk      = SkTree(max_depth=4, random_state=42).fit(X_tr, y_tr)

acc_dt_scratch = np.mean(dt_scratch.predict(X_te) == y_te)
acc_dt_sk      = np.mean(dt_sk.predict(X_te) == y_te)
print(f"Decision tree scratch accuracy: {acc_dt_scratch:.3f}")
print(f"Decision tree sklearn accuracy: {acc_dt_sk:.3f}")


## 4. Gaussian Naive Bayes

Assumes features are conditionally independent given class and Gaussian-distributed:
$$P(x_j | y=k) = \mathcal{N}(x_j; \mu_{jk}, \sigma_{jk}^2)$$

Use log-probabilities to avoid underflow when multiplying many small probabilities.

> 💡 **Interview Tip:** Amazon and enterprise companies often ask GNB. Key follow-up: *"When does the independence assumption hurt you?"* — when features are correlated (e.g., word co-occurrences in NLP). And: *"Why Gaussian? What if features are binary?"* — use Bernoulli NB instead.


In [ ]:
class GaussianNaiveBayes:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_  = {}
        self.means_   = {}
        self.vars_    = {}
        n = len(y)
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / n
            self.means_[c]  = X_c.mean(axis=0)
            self.vars_[c]   = X_c.var(axis=0) + 1e-9  # Laplace smoothing for variance
        return self

    def _log_likelihood(self, X, c):
        mu  = self.means_[c]
        var = self.vars_[c]
        # log N(x; mu, var) = -0.5*log(2*pi*var) - 0.5*(x-mu)^2/var
        return np.sum(-0.5 * np.log(2 * np.pi * var) - 0.5 * (X - mu)**2 / var, axis=1)

    def predict(self, X):
        log_posteriors = np.column_stack([
            np.log(self.priors_[c]) + self._log_likelihood(X, c)
            for c in self.classes_
        ])
        return self.classes_[log_posteriors.argmax(axis=1)]

# --- Test ---
from sklearn.naive_bayes import GaussianNB
gnb_scratch = GaussianNaiveBayes().fit(X_tr, y_tr)
gnb_sk      = GaussianNB().fit(X_tr, y_tr)

acc_gnb_scratch = np.mean(gnb_scratch.predict(X_te) == y_te)
acc_gnb_sk      = np.mean(gnb_sk.predict(X_te) == y_te)
print(f"GNB scratch accuracy: {acc_gnb_scratch:.3f}")
print(f"GNB sklearn accuracy: {acc_gnb_sk:.3f}")

# Summary
print("\n=== Summary ===")
for name, acc in [("KNN", acc_scratch), ("Decision Tree", acc_dt_scratch), ("GNB", acc_gnb_scratch)]:
    print(f"  {name:<15}: {acc:.3f}")


## Interview Frequency by Company Type

| Algorithm | Google | Meta | Amazon | Apple | Startups |
|---|---|---|---|---|---|
| KNN | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ | ⭐ |
| K-Means | ⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| Decision Tree | ⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ |
| Naive Bayes | ⭐ | ⭐ | ⭐⭐ | ⭐ | ⭐ |
| All + variants | | | | | ⭐⭐⭐ |

Google/Meta lean toward GD-based and neural implementations. Amazon/Apple ask more classical algorithms. Startups want breadth.

## Common Interview Questions

**Q: How do you compute pairwise distances efficiently?**
Use the identity $\|a-b\|^2 = \|a\|^2 - 2a^Tb + \|b\|^2$. This reduces a double loop to one matrix multiplication (O(n·m·d)) plus two norm computations. For approximate nearest neighbor at scale, switch to FAISS or HNSW.

**Q: What is K-means++ and why does it help?**
K-means++ initializes centroids by sampling with probability proportional to squared distance from existing centroids. This spreads them out and avoids the pathological case where all centroids start near each other. In practice it reduces iterations needed to converge and finds better local optima.

**Q: How do you handle continuous features in a decision tree split?**
Sort the feature values and evaluate the midpoint between each adjacent pair as a candidate threshold. This is O(n log n) per feature per split. sklearn uses a slightly more efficient version that pre-sorts once.

**Q: Why use log-probabilities in Naive Bayes?**
Multiplying many small probabilities underflows to 0 in floating point. Adding log-probabilities is equivalent but numerically stable. The predicted class is the argmax of log-posteriors, which is equivalent to argmax of posteriors.

**Q: What are the failure modes of K-means?**
Assumes convex spherical clusters of roughly equal size. Fails on elongated clusters, concentric rings, or when cluster sizes differ wildly. Local optima sensitivity is reduced by K-means++ and multiple restarts. Use DBSCAN or Gaussian Mixture Models for non-spherical clusters.

## Key Takeaways
- KNN has no training; all cost is at inference. The `a²-2ab+b²` trick is the vectorization interview answer
- K-means++ initialization is always preferred; it's one extra paragraph of code for much better results
- Decision tree splitting is O(n·d·log n) — most time is sorting feature values to find thresholds
- Naive Bayes uses log-probabilities to avoid underflow; always implement in log-space
- Know the complexity of each algorithm; interviewers ask this immediately after you implement
- The follow-up to every algorithm is "how does it scale?" — have ANN, random forests, or probabilistic alternatives ready